# Interrupt, resume, compact

A long-running agent may be stopped by a crash, deploy, rate limit, or human pause. Its useful work should survive.

In this notebook, we will:

- save state after every model step;
- simulate a crash;
- resume from the saved state; and
- compact old history before continuing.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

We use one return ticket and the normal tool loop. First, we run it without interruption so we know the expected result.

In [ ]:
import shoplab.llm
from shoplab.controls import Checkpoint
from shoplab.loop import run_agent
from shoplab.tools import Ledger, run_tool, standard_tools, to_openai_tools
from shoplab.world import load_tickets

tickets = {
    item["ticket_id"]: item
    for rows in load_tickets().values()
    for item in rows
}
ticket = tickets["TKT-2228"]
task = json.dumps({key: value for key, value in ticket.items() if key != "gold"})

SYSTEM = (
    "Handle this return. Look up the order, customer, and policy. "
    "Carry out the result, then call finish with decision, policy_id, and refund_usd."
)

call_start = len(shoplab.llm.LEDGER)
normal = run_agent(
    task,
    standard_tools(),
    model=MODEL,
    system=SYSTEM,
    max_steps=10,
)
normal_calls = len(shoplab.llm.LEDGER) - call_start

print("normal answer:", normal.answer)
print("model calls:", normal_calls)

## 1. Save after every step

The function below is the same basic tool loop, with one extra action: save messages and step after every model turn.

In [ ]:
class SimulatedCrash(RuntimeError):
    pass

def drive(messages, step, tools, checkpoint_path, max_steps=10, crash_after=None):
    answer = None

    while step < max_steps and answer is None:
        step += 1
        message = shoplab.llm.complete(
            messages,
            model=MODEL,
            tools=to_openai_tools(tools),
        ).choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        for request in message.tool_calls or []:
            arguments = json.loads(request.function.arguments or "{}")
            output = run_tool(tools, request.function.name, arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": request.id,
                "content": output,
            })

            if request.function.name == "finish":
                answer = arguments

        Checkpoint.save({
            "messages": messages,
            "step": step,
            "meta": {"finished": answer is not None},
        }, checkpoint_path)

        if crash_after == step and answer is None:
            raise SimulatedCrash(f"stopped after step {step}")

    return {"answer": answer, "step": step, "messages": messages}

A small runner starts with a new message list or resumes from a checkpoint.

In [ ]:
class DurableRunner:
    def __init__(self, path, tools, system, max_steps=10):
        self.path = Path(path)
        self.tools = tools
        self.system = system
        self.max_steps = max_steps

    def start(self, task, crash_after=None):
        messages = [
            {"role": "system", "content": self.system},
            {"role": "user", "content": task},
        ]
        return drive(
            messages,
            0,
            self.tools,
            self.path,
            max_steps=self.max_steps,
            crash_after=crash_after,
        )

    def resume(self):
        saved = Checkpoint.load(self.path)
        return drive(
            saved["messages"],
            saved["step"],
            self.tools,
            self.path,
            max_steps=self.max_steps,
        )

## 2. Simulate a crash

We stop after step two. The Python call ends, but the last completed state remains on disk.

In [ ]:
checkpoint_path = course_root / "artifacts/checkpoints/simplified-12-crash.json"
crash_ledger = Ledger()
runner = DurableRunner(
    checkpoint_path,
    standard_tools(crash_ledger),
    SYSTEM,
)

try:
    runner.start(task, crash_after=2)
except SimulatedCrash as error:
    print("crash:", error)

saved = Checkpoint.load(checkpoint_path)
print("saved step:", saved["step"])
print("saved messages:", len(saved["messages"]))
print("side effects before crash:", crash_ledger.entries)

## 3. Resume, do not restart

Resume begins with the saved messages. The model can see tool results already collected, so it does not need to repeat those lookups.

In [ ]:
resume_start = len(shoplab.llm.LEDGER)
resumed = runner.resume()
resume_calls = len(shoplab.llm.LEDGER) - resume_start

print("resumed answer:", resumed["answer"])
print("normal answer: ", normal.answer)
print("resume model calls:", resume_calls)
print("cold run model calls:", normal_calls)

A real system must also make side effects idempotent. If a refund succeeded just before a crash, retrying it must not issue a second refund. Checkpoints protect conversation state; payment systems need their own duplicate protection.

## 4. Compact before continuing a long session

A saved conversation can still become too large. We can keep recent turns and replace older turns with one summary.

In [ ]:
from shoplab.context import compact, count_tokens

session = Checkpoint.load(checkpoint_path)
before_tokens = count_tokens(session["messages"], model=MODEL)
short_messages = compact(session["messages"], keep_last=4)
after_tokens = count_tokens(short_messages, model=MODEL)

compact_path = course_root / "artifacts/checkpoints/simplified-12-compact.json"
Checkpoint.save({
    "messages": short_messages,
    "step": session["step"],
    "meta": {"compacted": True},
}, compact_path)

print("before:", len(session["messages"]), "messages,", before_tokens, "tokens")
print("after: ", len(short_messages), "messages,", after_tokens, "tokens")

## What different interruptions need

| Interruption | What should survive |
|---|---|
| Process crash | Last completed checkpoint |
| Rate limit | Messages plus a later retry time |
| Human pause | Messages plus the pending approval |
| Deployment | Durable checkpoint outside the process |
| Context overflow | Checkpoint plus compacted history |

## What to remember

1. Save after every completed model step.
2. A checkpoint contains messages, step, and metadata.
3. Resume from the checkpoint instead of starting over.
4. Make side effects safe to retry.
5. Compact old history when a durable session grows too large.

## Small practice steps

1. Crash after step one instead of step two.
2. Add the ticket id to checkpoint metadata.
3. Change keep_last and compare token counts.